# MEIKA Vox · transcripción desde Google Drive

**Paso 1:** prepara el motor una vez por sesión (instalación condicional y tiempos visibles).
**Paso 2:** conecta tu Google Drive y abre el panel. Si falla Drive, repite solo el paso 2.
**Paso 3:** selecciona los audios y el modo de transcripción. Con GPU se usa WhisperX large-v3.

El sistema es gratuito y de código abierto. Las transcripciones deben revisarse antes de citarse.


In [ ]:
#@title 1 · Preparar MEIKA Vox (una vez por sesión)
import importlib
import importlib.metadata as metadata
import shutil
import subprocess
import sys
import threading
import time
import tomllib
from pathlib import Path

if any(t.name in {"MEIKA_Vox_batch", "MEIKA_Vox_search"} and t.is_alive()
       for t in threading.enumerate()):
    raise RuntimeError("Hay un lote activo. Espera a que termine.")

REPO = Path("/content/meika-vox")
REPO_REF = "fix/colab-engine-stability-20261009"
REPORT = Path("/content/meika_vox_preparacion.txt")
_started = time.perf_counter()
_timings = []
_state = "iniciando"

def report():
    rows = ["MEIKA Vox · preparación",
            f"Python: {sys.version.split()[0]}",
            f"Repositorio: {REPO_REF}",
            f"Estado: {_state}"]
    rows.extend(f"{label}: {elapsed:.2f} s" for label, elapsed in _timings)
    rows.append(f"Total: {time.perf_counter() - _started:.2f} s")
    REPORT.write_text("\n".join(rows) + "\n", encoding="utf-8")

def timed(label, action):
    global _state
    _state = label
    report()
    print(f"Preparando: {label}…", flush=True)
    t = time.perf_counter()
    try:
        return action()
    finally:
        elapsed = time.perf_counter() - t
        _timings.append((label, elapsed))
        print(f"{label}: {elapsed:.1f} s", flush=True)
        report()

def checked(command, *, live=False):
    args = [str(x) for x in command]
    if live:
        result = subprocess.run(args, check=False)
    else:
        result = subprocess.run(args, capture_output=True, text=True, check=False)
    if result.returncode:
        if not live:
            print(result.stderr[-6000:] or result.stdout[-6000:])
        raise RuntimeError("Falló la preparación; revisa el error anterior.")

def prepare_tools():
    missing = [name for name in ("ffmpeg", "ffprobe", "git") if not shutil.which(name)]
    if not missing:
        print("FFmpeg, ffprobe y Git ya están disponibles.")
        return
    checked(["apt-get", "-qq", "update"])
    checked(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"])

def update_code():
    if not (REPO / ".git").exists():
        checked(["git", "clone", "--quiet", "https://github.com/MeikaLab/meika-vox.git", REPO])
    checked(["git", "-C", REPO, "fetch", "--quiet", "origin", REPO_REF])
    checked(["git", "-C", REPO, "checkout", "--quiet", "--detach", "FETCH_HEAD"])

def prepare_packages():
    # Use the project's version pins: never bypass compatibility with --no-deps.
    project = tomllib.loads((REPO / "pyproject.toml").read_text(encoding="utf-8"))
    packages = project["project"]["optional-dependencies"]["whisperx"]
    exact = {}
    for item in packages:
        if "==" in item:
            name, version = item.split("==", 1)
            exact[name] = version
    missing = []
    for name, version in exact.items():
        try:
            installed = metadata.version(name).split("+")[0]
        except metadata.PackageNotFoundError:
            installed = None
        if installed != version:
            missing.append(f"{name}: {installed or 'sin instalar'} → {version}")
    try:
        metadata.version("meika-vox")
        metadata.version("ipywidgets")
    except metadata.PackageNotFoundError:
        missing.append("MEIKA Vox / controles del panel")
    if not missing:
        print("Dependencias compatibles: instalación omitida.")
        return
    print("Instalando versiones necesarias:", "; ".join(missing), flush=True)
    loaded = set(sys.modules).intersection({
        "torch", "torchaudio", "torchvision", "transformers", "numpy"
    })
    if loaded:
        raise RuntimeError(
            "Hay módulos anteriores cargados en memoria: " + ", ".join(sorted(loaded))
            + ". Reinicia la sesión de Colab antes de actualizar dependencias."
        )
    checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "-e", str(REPO) + "[whisperx]", "ipywidgets>=8,<9"], live=True)

def check_engine():
    # A fresh interpreter catches the real transformers.Pipeline import failure.
    global probe
    probe = subprocess.run([sys.executable, "-c", """
import torch, transformers, whisperx
import whisperx.asr, whisperx.alignment
from transformers import Pipeline
print("WhisperX listo:", whisperx.__file__)
"""], capture_output=True, text=True, timeout=180)
    if probe.returncode:
        print(probe.stderr[-12000:] or probe.stdout[-12000:])
        raise RuntimeError("El motor no pudo importarse. Revisa el error anterior.")
    for name in list(sys.modules):
        if name == "meika_vox" or name.startswith("meika_vox."):
            del sys.modules[name]
    if str(REPO / "src") not in sys.path:
        sys.path.insert(0, str(REPO / "src"))
    runtime = importlib.import_module("meika_vox.runtime").inspect_runtime()
    if not runtime.asr_ready:
        raise RuntimeError("Instalación incompleta de WhisperX o FFmpeg.")
    print("Motor listo. Si falla Drive, vuelve a ejecutar SOLO la celda 2.")

try:
    timed("FFmpeg y Git", prepare_tools)
    timed("Repositorio", update_code)
    timed("Dependencias", prepare_packages)
    timed("Comprobación real del motor", check_engine)
except Exception:
    _state = "falló"
    report()
    print("Diagnóstico disponible en", REPORT)
    raise
_state = "preparado"
report()
print(f"PREPARACIÓN LISTA · {time.perf_counter() - _started:.1f} s · informe: {REPORT}")
print("Continúa con «2 · Conectar Drive y abrir el panel».")


In [ ]:
#@title 2 · Conectar Drive y abrir MEIKA Vox (repetible)
import importlib.util
import threading
import time
from pathlib import Path

from google.colab import drive, output

REPO = Path("/content/meika-vox")
if not (REPO / "src/meika_vox").is_dir():
    raise RuntimeError("Primero ejecuta «1 · Preparar MEIKA Vox».")
if any(t.name in {"MEIKA_Vox_batch", "MEIKA_Vox_search"} and t.is_alive()
       for t in threading.enumerate()):
    raise RuntimeError("Hay una transcripción en curso; espera a que termine.")

start_drive = time.perf_counter()
print("Conectando Google Drive…", flush=True)
try:
    drive.mount("/content/drive")
except ValueError as exc:
    print("No se pudo montar Drive. El motor YA está preparado.")
    print("Comprueba la autorización de la cuenta Google y vuelve a pulsar")
    print("SOLO la celda 2; NO reinstales el motor ni ejecutes de nuevo la celda 1.")
    print("Si vuelve a fallar, reinicia la sesión y vuelve a autorizar Drive.")
    raise RuntimeError("Falló la autorización/montaje de Google Drive.") from exc

root = Path("/content/drive/MyDrive")
if not root.is_dir():
    raise RuntimeError("Drive se montó, pero no aparece Mi unidad. Revisa la cuenta autorizada.")
print(f"Drive conectado en {time.perf_counter() - start_drive:.1f} s")
output.enable_custom_widget_manager()
spec = importlib.util.spec_from_file_location("vox_ui", REPO / "scripts/colab_ui.py")
ui = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ui)
print(f"Drive verificado; preparando panel: {time.perf_counter() - start_drive:.1f} s")
_panel = ui.build_panel(REPO, root)
